# Conclusão e entrega reproduzível — FD001

[English](../en-US/27_conclusion_delivery_fd001.ipynb) · [README em português](../../README.md)

A etapa FD001 foi concluída com um Extra Trees, otimizado com validação por motor e treinado com features causais. A representação de entrada possui 205 atributos; 93 foram selecionados pelo pipeline final. Esta edição em português é canônica. A lógica reutilizável é mantida em `src`, e a inferência pode ser executada pelos scripts da entrega.

## Origem dos dados e significado de NASA

O conjunto utilizado é o **Turbofan Engine Degradation Simulation Data Set**, disponibilizado pelo Prognostics Center of Excellence (PCoE) do NASA Ames Research Center (Saxena e Goebel, 2008). NASA é a sigla de **National Aeronautics and Space Administration**, a agência espacial e aeronáutica dos Estados Unidos. No título deste projeto, ela identifica a origem dos dados; a análise e o modelo são trabalhos autorais de portfólio. [Repositório oficial da NASA](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/).

As séries foram geradas com o simulador **C-MAPSS — Commercial Modular Aero-Propulsion System Simulation**, que representa a operação e a degradação de motores aeronáuticos turbofan comerciais. São dados de simulação: no treinamento, cada trajetória acompanha um motor até a falha; no teste, o histórico termina antes da falha e o RUL verdadeiro do último ciclo é fornecido separadamente. [Catálogo oficial do dataset](https://data.nasa.gov/dataset/cmapss-jet-engine-simulated-data).

Foi utilizado o subconjunto **FD001**, com 100 motores de treinamento e 100 de teste, uma condição operacional e um modo de falha: degradação do compressor de alta pressão (HPC). Essa configuração permite estudar previsão de vida restante sob condições controladas; não demonstra desempenho em uma frota real.

**Referência do dataset:** Saxena, A.; Goebel, K. (2008). *Turbofan Engine Degradation Simulation Data Set*. NASA Ames, Prognostics Data Repository. [Fonte e crédito original](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/).

Insights: A atribuição identifica os responsáveis pela disponibilização dos dados e distingue a origem do benchmark da autoria desta análise. Como FD001 é simulado, os resultados precisam ser interpretados dentro desse cenário.

## Resultado do benchmark

| Modelo | MAE | RMSE | Erro médio assinado |
| --- | ---: | ---: | ---: |
| Extra Trees | 16,23 | 23,80 | +10,08 |
| XGBoost | 17,29 | 25,15 | +9,31 |
| HistGradientBoosting | 17,77 | 27,01 | +8,83 |

Insights: Os candidatos foram avaliados nos mesmos 100 motores do teste oficial, após o congelamento de hiperparâmetros. O Extra Trees foi escolhido após essa comparação e apresentou menor MAE e RMSE. O benchmark não constitui uma avaliação independente adicional após a escolha do vencedor. A superestimação média de 10,08 ciclos deverá ser considerada ao interpretar o risco de manutenção. Os valores foram transcritos do registro congelado da etapa 21; não foi realizada outra avaliação no teste para esta entrega.

## Representação e explicabilidade

As médias e desvios-padrão foram calculados nas janelas de 5, 10, 20, 40 e 80 ciclos, sempre dentro de cada motor e incluindo somente observações disponíveis até o ciclo da previsão. Deltas foram oferecidos à seleção, mas nenhum foi mantido no ajuste final. Foram selecionados 13 atributos-base, 60 médias móveis e 20 desvios-padrão móveis.

Insights: O ciclo observado foi o atributo de maior importância por permutação; os desvios-padrão em janelas longas também apresentaram contribuição relevante. Esses resultados indicam dependência preditiva do histórico dos sensores, sem estabelecer causalidade física. Features correlacionadas podem dividir importância.

## Utilidade operacional e limites

Foi observado que o limiar ilustrativo de RUL previsto igual a 20 ciclos gerou alertas oportunos nas 100 trajetórias OOF, após confirmação por três ciclos e sob uma janela útil de 5 a 30 ciclos. A antecedência mediana foi de 16 ciclos. Os cenários de custo foram hipotéticos e analisados apenas no desenvolvimento.

Insights: O resultado permite demonstrar como uma previsão pode alimentar uma política de manutenção, mas não valida a implantação. Os folds foram usados durante a otimização, e o teste oficial contém trajetórias truncadas. A operação real exige custos e antecedência mínima definidos pelos responsáveis pela manutenção.

Na análise exploratória de incerteza, a cobertura média dos ciclos por motor foi de 85,9%, a largura média foi de 91,3 ciclos e 46% das trajetórias foram inteiramente cobertas. Para RUL previsto acima de 100 ciclos, a cobertura média foi de 76,4%.

Insights: O intervalo global ficou amplo e não atingiu a referência de calibração em todos os regimes. A entrega conterá somente a previsão pontual; os intervalos OOF não serão apresentados como intervalos validados do modelo refitado.

## Entrega e reprodução

O pipeline congelado será encontrado em `../../outputs/models/extra_trees_fd001_v1/model.joblib`, com parâmetros, atributos, hashes, versões e verificações em `metadata.json`. O histórico observado será fornecido como CSV e processado por `../../scripts/predict_final_extra_trees_fd001.py`. Será exigido histórico contínuo desde o ciclo 1, com identificadores, ciclo e 17 sinais, conforme contrato descrito no [README](../../README.md).

O exemplo incluído utiliza um prefixo do treinamento e serve como teste da interface. As verificações da entrega comparam previsões antes e depois da serialização, conferem a engenharia de features, verificam que ciclos futuros não alteram previsões passadas e rejeitam ciclos duplicados ou históricos incompletos.

Insights: A entrega permite executar uma previsão local sem repetir o Optuna. Os testes verificam o funcionamento do pipeline e o contrato temporal, sem acrescentar uma nova estimativa de desempenho fora da amostra.

## Percurso de publicação

A leitura principal poderá ser iniciada por esta síntese, seguindo para a ablação temporal, triagem, otimização, benchmark, decisão e análise dos erros, explicabilidade, alertas e incerteza. Os notebooks restantes poderão ser preservados como histórico de experimentação. Não será necessário publicar 27 notebooks como uma sequência obrigatória de leitura.

Insights: A narrativa principal foi separada do histórico para tornar a avaliação do portfólio mais acessível. A organização está documentada no README; nenhum experimento foi removido. O README, o relatório HTML e o notebook 27 possuem versões em português e inglês; os demais notebooks permanecem na edição canônica em português. A publicação externa e a extensão para FD002–FD004 permanecem trabalhos posteriores.